# EDA dan Preprocessing
Analisis data anak stunting Temanggung: eksplorasi data, pembersihan, penanganan outlier, binning, dan normalisasi fitur.

## Import Library

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

from utils import handle_outliers_iqr, bin_features


## Memuat Data Set

In [ ]:
df = pd.read_csv('../data/raw/data_anak_stunting_temanggung.csv')
print("=== 5 baris data set pertama ===")
df.head()


## Exploratory Data Analysis (EDA)

In [ ]:
print("==== Info dataset =====")
df.info()


In [ ]:
sns.pairplot(df, hue='jenis_kelamin')
plt.show()


In [ ]:
# --- Ubah tanggal_lahir menjadi tahun lahir ---
df['tanggal_lahir'] = pd.to_datetime(df['tanggal_lahir'], errors='coerce')
df['tahun_lahir'] = df['tanggal_lahir'].dt.year

# --- Ubah jenis_kelamin menjadi angka (1 = Laki-laki, 0 = Perempuan) ---
df['jenis_kelamin'] = df['jenis_kelamin'].map({'Laki-laki': 1, 'Perempuan': 0})

df[['jenis_kelamin', 'tanggal_lahir', 'tahun_lahir']].head()


In [ ]:
print("==== korelasi dataset ====")
numeric_df = df.select_dtypes(include=np.number)
numeric_df.corr()


In [ ]:
print("==== Deskripsi dataset =====")
df.describe()


In [ ]:
print("==== Duplikasi dataset =====")
print(df.duplicated().sum())

print("==== jumlah missing value ====")
print(df.isnull().sum())


In [ ]:
print("==== Cek Outlier Pada Data ====")
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_columns):
    plt.subplot(3, 4, i + 1)
    sns.boxplot(x=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title('Korelasi Antar Fitur')
plt.show()


## Data Processing

In [ ]:
print("=== menghapus data duplikat ===")
df.drop_duplicates(inplace=True)

print("=== menghapus missing value ===")
df.dropna(inplace=True)


In [ ]:
cols_with_outliers = ["usia_bulan", "tinggi_badan_cm", "skor_z_haz"]
df_cleaned, outlier_summary = handle_outliers_iqr(df, cols_with_outliers)
outlier_summary


In [ ]:
df_binned = bin_features(df)
df_binned[['usia_bulan', 'usia_bulan_bin', 'tinggi_badan_cm', 'tinggi_badan_cm_bin', 'skor_z_haz', 'skor_z_haz_bin']].head(10)


In [ ]:
print(df_binned['usia_bulan_bin'].value_counts())
print(df_binned['skor_z_haz_bin'].value_counts())
sns.countplot(data=df_binned, x='skor_z_haz_bin', hue='jenis_kelamin')
plt.show()


In [ ]:
print("=== normalisasi data ===")
numeric_features = ['usia_bulan', 'tinggi_badan_cm', 'skor_z_haz']

df[numeric_features] = df[numeric_features].apply(pd.to_numeric, errors='coerce')

scaler = MinMaxScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])
df[numeric_features].head()


## Simpan Data Bersih
Data yang sudah dibersihkan disimpan untuk dipakai di notebook pemodelan (`02_modeling.ipynb`), atau bisa juga langsung memakai `src/data_preprocessing.py` yang merangkum seluruh langkah di atas.

In [ ]:
df.to_csv('../data/processed/data_cleaned.csv', index=False)
print("Data bersih disimpan di data/processed/data_cleaned.csv")
